# Module 6.1: Supervised Fine-Tuning (SFT)

Welcome to the final phase! Our model has finished its massive Pre-Training loop (Module 5). It is now a **Base Model** (like Llama 3 8B Base). It perfectly understands languages, logic, and coding, but it doesn't know how to act like an assistant.

If you prompt it with: `"What is the capital of France?"`, it might respond with `"What is the capital of Germany?"` because it simply completes web-style text.

In this notebook, we look at **Supervised Fine-Tuning**, teaching our model purely how to chat using specialized Prompt Architectures.

## 1. Chat Templates (The Secret Code)

To make the model behave, we construct a strict format of hidden syntax tokens. The model learns that when it sees a specific syntax, it is expected to answer rather than just autocomplete.

A popular format (used by OpenAI's `ChatML` and Llama 3) uses special control tokens:
1. `<|im_start|>`: Marks the beginning of a role.
2. `<|im_end|>`: Marks the end of a message.
3. `<|system|>` / `<|user|>` / `<|assistant|>`: Defines who is speaking.

Let's see what user messages ACTUALLY look like before they enter the model.

In [ ]:
def apply_chat_template(user_message, system_message="You are a helpful assistant."):
    """
    Converts human-readable messages into the strict ChatML formatting 
    required for a Fine-Tuned model.
    """
    formatted_prompt = (
        f"<|im_start|>system\n{system_message}<|im_end|>\n"
        f"<|im_start|>user\n{user_message}<|im_end|>\n"
        f"<|im_start|>assistant\n" # We leave this open so the model completes it!
    )
    return formatted_prompt

# Let's test it!
raw_input = "What is the capital of France?"
sft_input = apply_chat_template(raw_input)

print("--- RAW INPUT ---")
print(raw_input)
print("\n--- WHAT THE MODEL ACTUALLY SEES ---")
print(sft_input)

## 2. Training the SFT Model (Masking the Context)

During Supervised Fine-Tuning, we create a dataset of thousands of perfect conversational examples. We pass the *entire* conversation (User message + Perfect Assistant reply) into the model using the Cross-Entropy Loss from Module 5.

**HOWEVER (The Golden Rule)**: We only calculate Loss on the *Assistant's* tokens! 
If we train the model to predict the human's tokens, it will try to predict what the user is going to say next, which ruins generation. We must explicitly `mask` the human instructions so the gradients are only applied to the Assistant's responses!

In [ ]:
import torch

# 1. Let's mock a sequence of 8 tokens in a conversation:
# (User) "Hi" -> [101]
# (Model) "Hello, how are you?" -> [50, 22, 90, 800]
tokens = torch.tensor([101, 50, 22, 90, 800])

# 2. We create a mask where User tokens are ignored (-100 is the PyTorch ignore index for Cross Entropy)
ignore_index = -100
target_labels = torch.tensor([ignore_index, 50, 22, 90, 800])

print(f"Original Sequence:   {tokens.tolist()}")
print(f"Target for Training: {target_labels.tolist()}")
print("-> PyTorch will silently enforce that loss is ONLY calculated for tokens over 0!")

## Summary

Supervised Fine-Tuning forces the model to adhere to a strict Question/Answer format. 

However, this doesn't prevent the model from being rude, saying "I don't know" immediately, or producing unsafe content. SFT just teaches it the format, not *values*. 

To instill human values, we move to the final piece of the modern AI pipeline: **Module 6.2: DPO (Preference Alignment)**!